# Customer Segments - Business Metrics

Segments customers into High/Mid/Low value tiers and Active/At Risk/Churned status. Answers:
- How many customers are in each value segment?
- Which customers are at risk of churning?
- What's the revenue distribution across segments?
- Who needs retention campaigns?

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.agg_customer_segments AS
WITH percentiles AS (
  SELECT 
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY lifetime_value) AS p75,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY lifetime_value) AS p25
  FROM workspace.gold.agg_customer_lifetime_value
)
SELECT 
  clv.*,
  CASE 
    WHEN clv.lifetime_value >= p.p75 THEN 'High Value'
    WHEN clv.lifetime_value >= p.p25 THEN 'Mid Value'
    ELSE 'Low Value'
  END AS customer_segment,
  CASE
    WHEN clv.days_since_last_order <= 30 THEN 'Active'
    WHEN clv.days_since_last_order <= 90 THEN 'At Risk'
    ELSE 'Churned'
  END AS customer_status
FROM workspace.gold.agg_customer_lifetime_value clv
CROSS JOIN percentiles p

In [0]:
%sql
SELECT 
  customer_segment,
  customer_status,
  COUNT(*) AS customer_count,
  SUM(lifetime_value) AS total_revenue,
  ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value
FROM workspace.gold.agg_customer_segments
GROUP BY customer_segment, customer_status
ORDER BY customer_segment, customer_status

In [0]:
%sql
SELECT 
  CONCAT(first_name, ' ', last_name) AS customer_name,
  country,
  lifetime_value,
  total_orders,
  days_since_last_order,
  customer_segment,
  customer_status
FROM workspace.gold.agg_customer_segments
WHERE customer_segment = 'High Value' AND customer_status IN ('At Risk', 'Churned')
ORDER BY lifetime_value DESC
LIMIT 20